In [ ]:
# !pip install pdfplumber
# !pip install PyMuPDF

In [ ]:
#Importing required libraries
import traceback
import pandas as pd
import numpy as np
import re
import pdfplumber
from collections import namedtuple
from calendar import month_name
import fitz

In [ ]:
import sys
print(sys.executable)
print(sys.version)
print(sys.version_info)

In [ ]:
#Import PDF file from path  
InputDate = '2026-07-20'
path = "D:/Merin/Task 7/LCR Playbook 2021 4A.pdf"
file = 'D:/Users/thomas2/Atwmchoi2010_GDrive/Projects/Neural Matrix Investment/LCR/LCR top20 2022-05-23.pdf'
file="G:\\My Drive\\projects\\Neural Matrix Investment\\LCR\\LCR top20 2022-04-18.pdf"
currentLCR = f"912 Top20 {InputDate}.pdf"
file=f"C:\\Users\\thomas2.DESKTOP-F01LKSM\\Google Drive Streaming\\My Drive\\projects\\Neural Matrix Investment\\LCR\\{currentLCR}"
file=f"C:\\Users\\Asus\\My Drive\\projects\\Neural Matrix Investment\\LCR\\{currentLCR}"
file=f"I:\\My Drive\\ReadProjects\\Neural Matrix Investment\\LCR\\{currentLCR}"
#   Ubuntu/WSL file path location
file=f"/mnt/i/My Drive/ReadProjects/Neural Matrix Investment/LCR/{currentLCR}"
print('Processing LCR:',file)
with fitz.open(file) as doc:
    text = ""
    text_full = ""
    c=0
    for page in doc:
        print("page type: ", type(page))
        print(f" in c:{c}, page is ", page)
        c += 1
        if c < 3: continue
        # text = page.getText()
        text = page.get_text()
        text_full += text
print(text_full)

In [ ]:
iterates = iter(text_full.split('\n'))
data_foo = [['','','','','']]

#Basic columns to be read from PDF
col= ["Symbol", "Status", "Expiration", "PnC", "Price" , "Entry1", "Entry2", "Target", "Stop"]
data_f = []
for x in col:
    data_f.append('')
data_foo = [data_f]
df = pd.DataFrame(data_foo,columns=col)

In [ ]:
Stock_name_tmp = None
Status = None
line_num = -1
row_number = -1
for line in iterates:
        line_num += 1
        print("line {}> {}".format(line_num, line))
        if line.strip() == '':
            continue
        if re.search("\[\w*\]", line):
            Stock_name_tmp = (re.findall(r"\[\w*\]", line)[0][1:-1])
            print("{},{}> find stock name: {}".format(line_num, row_number, Stock_name_tmp))
            Status = re.findall(r"\S*", next(iterates))[0]
        else:
            if (line.find("Aggressive Traders:") > -1):
                pass
            elif (line.find("Entry @ ") > -1) or (line.find("range of") > -1):
                df.loc[len(df.index)] = data_f
                row_number+= 1
                df['Symbol'][row_number] = Stock_name_tmp
                df['Status'][row_number] = Status
                entry = line.split('@')[1]
                mtch_entry = re.finditer(r"\d+(\.)?\d+", entry)
                ent = []
                for n in mtch_entry:
                    ent.append(n[0])
                ent1 = ent[0]
                ent2 = ent[1]
                df["Entry1"][row_number] = ent1
                df["Entry2"][row_number] = ent2
            elif (line.find("Target @ ") > -1):
                target = line.split('@')[1]
                mtch_trgt = re.finditer(r"\d+(\.)?\d+", target)
                trgt = []
                for z in mtch_trgt:
                    trgt.append(z[0])
                df["Target"][row_number] = trgt[0]
            elif ((line.find("STO") > -1) or (line.find("BTO") > -1)):
                decission = re.findall(r"Put|Call", line)[0][0]
                print("{}> {} from line={} found PnC={}".format(row_number, df['Symbol'][row_number], line, decission))
                exp_date = re.findall(r"\(...*\)", line)[0][1:-4]
                df['PnC'][row_number] = decission
                df['Expiration'][row_number] = exp_date
                price = line.split('(')[0]
                matches = re.finditer(r"\d+(\.)?\d+", price)
                tmp_price = []
                for m in matches:
                    tmp_price.append(m[0])
                prc = tmp_price[0]
                df["Price"][row_number] = prc
            elif (line.find("Stop") > -1):
                stop = line.split('@')[1]
                mtch_stop = re.finditer(r"\d+(\.)?\d+", stop)
                stp = []
                for q in mtch_stop:
                    stp.append(q[0])
                df["Stop"][row_number] = stp[0]
            else:
                pass
print(df)

In [ ]:
df = df[:-1]
df = df.fillna('')
df['Entry_Sign'] = df['PnC'].apply(lambda x: '<' if x == 'P' else '>')
df['Target_Sign'] = df['PnC'].apply(lambda x: '>' if x == 'P' else '<')
df['Stop_Sign'] = df['Entry_Sign']

In [ ]:
df

In [ ]:
df["Entry1"], df["Entry2"] = np.where(df["PnC"]=="C", [df["Entry2"], df["Entry1"]], [df["Entry1"], df["Entry2"] ])

In [ ]:
#Arranging columns as required in output file
df['Date'] = InputDate
df = df[["Date", "Symbol", "Status", "Expiration", "PnC", "Price" , "Entry_Sign", "Entry1", "Entry2", "Target_Sign", "Target", "Stop_Sign", "Stop"]]

df.rename(columns = {'Price':'Strike'}, inplace = True)
#Exports the final dataframe to txt file
df.to_csv(file.replace('.pdf', '.csv'), sep=',', index = None)

In [ ]:
from dotenv import load_dotenv
import logging
import numpy as np
import dataUtil as DU

In [ ]:
df.info()

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])
df["Expiration"] = pd.to_datetime(df["Expiration"])
df = df.replace(r'^\s*$', np.nan, regex=True)
# final_df['L_Strike'] = final_df['L_Strike'].astype(float)
df = df.astype({'Strike':'float','Entry1':'float','Entry2':'float','Target':'float','Stop':'float'})
df.info()

In [ ]:
df

In [ ]:
load_dotenv("DB_Config.env") #Check path for env variables
logging.getLogger().setLevel(logging.DEBUG)

In [ ]:
opt_tbl = "Stock_Options"
trd_DB = "Trading"
DU.StoreEOD(df, trd_DB, opt_tbl)